# 多智能体去中心化发言者选择

本笔记本展示了如何在没有固定发言时间表的情况下实现多智能体模拟。取而代之的是，智能体自行决定谁发言。我们可以通过让每个智能体竞标发言来实现这一点。出价最高的智能体将获得发言权。

在下面的示例中，我们将展示如何实现这一点，该示例展示了一场虚构的总统辩论。

## 导入 LangChain 相关模块

In [1]:
from typing import Callable, List

import tenacity
from langchain.output_parsers import RegexParser
from langchain.prompts import PromptTemplate
from langchain.schema import (
    HumanMessage,
    SystemMessage,
)
from langchain_openai import ChatOpenAI

## `DialogueAgent` 和 `DialogueSimulator` 类
我们将使用与[多人龙与地下城](https://python.langchain.com/en/latest/use_cases/agent_simulations/multi_player_dnd.html)中定义的相同的 `DialogueAgent` 和 `DialogueSimulator` 类。

In [2]:
class DialogueAgent:
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatOpenAI,
    ) -> None:
        self.name = name
        self.system_message = system_message
        self.model = model
        self.prefix = f"{self.name}: "
        self.reset()

    def reset(self):
        self.message_history = ["Here is the conversation so far."]

    def send(self) -> str:
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        message = self.model.invoke(
            [
                self.system_message,
                HumanMessage(content="\n".join(self.message_history + [self.prefix])),
            ]
        )
        return message.content

    def receive(self, name: str, message: str) -> None:
        """
        Concatenates {message} spoken by {name} into message history
        """
        self.message_history.append(f"{name}: {message}")


class DialogueSimulator:
    def __init__(
        self,
        agents: List[DialogueAgent],
        selection_function: Callable[[int, List[DialogueAgent]], int],
    ) -> None:
        self.agents = agents
        self._step = 0
        self.select_next_speaker = selection_function

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        """
        Initiates the conversation with a {message} from {name}
        """
        for agent in self.agents:
            agent.receive(name, message)

        # increment time
        self._step += 1

    def step(self) -> tuple[str, str]:
        # 1. choose the next speaker
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]

        # 2. next speaker sends message
        message = speaker.send()

        # 3. everyone receives message
        for receiver in self.agents:
            receiver.receive(speaker.name, message)

        # 4. increment time
        self._step += 1

        return speaker.name, message

## `BiddingDialogueAgent` 类
我们定义了一个 `DialogueAgent` 的子类，它有一个 `bid()` 方法，该方法根据消息历史和最近一条消息生成一个出价。

In [3]:
class BiddingDialogueAgent(DialogueAgent):
    def __init__(
        self,
        name,
        system_message: SystemMessage,
        bidding_template: PromptTemplate,
        model: ChatOpenAI,
    ) -> None:
        super().__init__(name, system_message, model)
        self.bidding_template = bidding_template

    def bid(self) -> str:
        """
        Asks the chat model to output a bid to speak
        """
        prompt = PromptTemplate(
            input_variables=["message_history", "recent_message"],
            template=self.bidding_template,
        ).format(
            message_history="\n".join(self.message_history),
            recent_message=self.message_history[-1],
        )
        bid_string = self.model.invoke([SystemMessage(content=prompt)]).content
        return bid_string

## 定义参与者和辩论主题

In [4]:
character_names = ["Donald Trump", "Kanye West", "Elizabeth Warren"]
topic = "transcontinental high speed rail"
word_limit = 50

## 生成系统消息

In [5]:
game_description = f"""Here is the topic for the presidential debate: {topic}.
The presidential candidates are: {', '.join(character_names)}."""

player_descriptor_system_message = SystemMessage(
    content="You can add detail to the description of each presidential candidate."
)


def generate_character_description(character_name):
    character_specifier_prompt = [
        player_descriptor_system_message,
        HumanMessage(
            content=f"""{game_description}
            Please reply with a creative description of the presidential candidate, {character_name}, in {word_limit} words or less, that emphasizes their personalities. 
            Speak directly to {character_name}.
            Do not add anything else."""
        ),
    ]
    character_description = ChatOpenAI(temperature=1.0)(
        character_specifier_prompt
    ).content
    return character_description


def generate_character_header(character_name, character_description):
    return f"""{game_description}
Your name is {character_name}.
You are a presidential candidate.
Your description is as follows: {character_description}
You are debating the topic: {topic}.
Your goal is to be as creative as possible and make the voters think you are the best candidate.
"""


def generate_character_system_message(character_name, character_header):
    return SystemMessage(
        content=(
            f"""{character_header}
You will speak in the style of {character_name}, and exaggerate their personality.
You will come up with creative ideas related to {topic}.
Do not say the same things over and over again.
Speak in the first person from the perspective of {character_name}
For describing your own body movements, wrap your description in '*'.
Do not change roles!
Do not speak from the perspective of anyone else.
Speak only from the perspective of {character_name}.
Stop speaking the moment you finish speaking from your perspective.
Never forget to keep your response to {word_limit} words!
Do not add anything else.
    """
        )
    )


character_descriptions = [
    generate_character_description(character_name) for character_name in character_names
]
character_headers = [
    generate_character_header(character_name, character_description)
    for character_name, character_description in zip(
        character_names, character_descriptions
    )
]
character_system_messages = [
    generate_character_system_message(character_name, character_headers)
    for character_name, character_headers in zip(character_names, character_headers)
]

In [6]:
for (
    character_name,
    character_description,
    character_header,
    character_system_message,
) in zip(
    character_names,
    character_descriptions,
    character_headers,
    character_system_messages,
):
    print(f"\n\n{character_name} Description:")
    print(f"\n{character_description}")
    print(f"\n{character_header}")
    print(f"\n{character_system_message.content}")



Donald Trump Description:

Donald Trump, you are a bold and outspoken individual, unafraid to speak your mind and take on any challenge. Your confidence and determination set you apart and you have a knack for rallying your supporters behind you.

Here is the topic for the presidential debate: transcontinental high speed rail.
The presidential candidates are: Donald Trump, Kanye West, Elizabeth Warren.
Your name is Donald Trump.
You are a presidential candidate.
Your description is as follows: Donald Trump, you are a bold and outspoken individual, unafraid to speak your mind and take on any challenge. Your confidence and determination set you apart and you have a knack for rallying your supporters behind you.
You are debating the topic: transcontinental high speed rail.
Your goal is to be as creative as possible and make the voters think you are the best candidate.


Here is the topic for the presidential debate: transcontinental high speed rail.
The presidential candidates are: Dona

## 出价的输出解析器
我们要求代理输出一个出价以进行发言。但由于代理是输出字符串的 LLM，我们需要：
1. 定义它们将生成输出的格式
2. 解析它们的输出

我们可以继承 [RegexParser](https://github.com/langchain-ai/langchain/blob/master/langchain/output_parsers/regex.py) 来实现我们自己的自定义出价输出解析器。

In [7]:
class BidOutputParser(RegexParser):
    def get_format_instructions(self) -> str:
        return "Your response should be an integer delimited by angled brackets, like this: <int>."


bid_parser = BidOutputParser(
    regex=r"<(\d+)>", output_keys=["bid"], default_output_key="bid"
)

## 生成竞价系统消息
这受到[Generative Agents](https://arxiv.org/pdf/2304.03442.pdf)中用于使用 LLM 确定记忆重要性的提示的启发。这将使用我们 `BidOutputParser` 的格式说明。

In [8]:
def generate_character_bidding_template(character_header):
    bidding_template = f"""{character_header}

```
{{message_history}}
```

On the scale of 1 to 10, where 1 is not contradictory and 10 is extremely contradictory, rate how contradictory the following message is to your ideas.

```
{{recent_message}}
```

{bid_parser.get_format_instructions()}
Do nothing else.
    """
    return bidding_template


character_bidding_templates = [
    generate_character_bidding_template(character_header)
    for character_header in character_headers
]

In [9]:
for character_name, bidding_template in zip(
    character_names, character_bidding_templates
):
    print(f"{character_name} Bidding Template:")
    print(bidding_template)

Donald Trump Bidding Template:
Here is the topic for the presidential debate: transcontinental high speed rail.
The presidential candidates are: Donald Trump, Kanye West, Elizabeth Warren.
Your name is Donald Trump.
You are a presidential candidate.
Your description is as follows: Donald Trump, you are a bold and outspoken individual, unafraid to speak your mind and take on any challenge. Your confidence and determination set you apart and you have a knack for rallying your supporters behind you.
You are debating the topic: transcontinental high speed rail.
Your goal is to be as creative as possible and make the voters think you are the best candidate.


```
{message_history}
```

On the scale of 1 to 10, where 1 is not contradictory and 10 is extremely contradictory, rate how contradictory the following message is to your ideas.

```
{recent_message}
```

Your response should be an integer delimited by angled brackets, like this: <int>.
Do nothing else.
    
Kanye West Bidding Templat

## 使用 LLM 创建关于辩论主题的详尽阐述

This guide will walk you through the process of using a large language model (LLM) to generate an elaborate on a debate topic. This can be a valuable tool for debaters, researchers, or anyone looking to explore a topic from multiple perspectives.

本指南将引导您完成使用大型语言模型 (LLM) 为辩论主题生成详尽阐述的过程。这对于辩手、研究人员或任何希望从多角度探讨主题的人来说，都是一个有价值的工具。

### 1. Choose Your Debate Topic

### 1. 选择您的辩论主题

The first step is to select a clear and debatable topic. A good debate topic is often one that has at least two distinct sides with valid arguments for each.

第一步是选择一个清晰且可辩论的主题。一个好的辩论主题通常是至少有两个截然不同且双方都有有效论据的方面。

**Examples:**

**示例：**

*   **Resolved:** Artificial intelligence will ultimately benefit humanity more than it harms it.
    **决定：** 人工智能最终将比损害人类更多地造福人类。
*   **Resolved:** Social media has a net negative impact on society.
    **决定：** 社交媒体对社会产生了净负面影响。
*   **Resolved:** Universal basic income is a viable solution to poverty.
    **决定：** 普遍基本收入是解决贫困的可行方案。

### 2. Craft Your Prompt

### 2. 精心设计您的提示词

The quality of your output heavily depends on the quality of your prompt. Be specific about what you want the LLM to generate.

输出的质量在很大程度上取决于提示词的质量。请具体说明您希望 LLM 生成的内容。

**Key elements to include in your prompt:**

**提示词中应包含的关键要素：**

*   **The debate topic:** Clearly state the resolution.
    **辩论主题：** 清晰陈述决议。
*   **The desired perspective:** Specify which side of the debate you want the LLM to elaborate on (e.g., affirmative/pro, negative/con).
    **期望的视角：** 指定您希望 LLM 阐述辩论的哪一方（例如，正方/支持方，反方/反对方）。
*   **The level of detail:** Indicate whether you want a brief overview, a detailed argument, or a comprehensive analysis.
    **详细程度：** 指示您是想要一个简要概述、一个详细论点还是一个全面分析。
*   **Specific aspects to cover:** You can ask the LLM to focus on particular arguments, evidence, counter-arguments, or potential rebuttals.
    **要涵盖的具体方面：** 您可以要求 LLM 关注特定的论点、证据、反驳论点或潜在的驳斥。
*   **Format:** Specify the desired output format (e.g., bullet points, paragraphs, a structured essay).
    **格式：** 指定期望的输出格式（例如，项目符号、段落、结构化论文）。

**Example Prompt:**

**示例提示词：**

```
Elaborate on the affirmative case for the resolution: "Resolved: Artificial intelligence will ultimately benefit humanity more than it harms it." Focus on economic benefits, advancements in healthcare, and improvements in quality of life. Provide specific examples and potential evidence to support these points. Structure the response with an introduction, body paragraphs for each benefit, and a concluding statement.
```

```
请详尽阐述关于“决定：人工智能最终将比损害人类更多地造福人类”这一决议的正方论点。重点关注经济效益、医疗保健领域的进步以及生活质量的提高。请提供具体示例和潜在证据来支持这些观点。请以引言、每个效益的主体段落以及结论性陈述来组织回应。
```

### 3. Interact with the LLM

### 3. 与 LLM 进行交互

Input your prompt into the LLM. You may need to iterate and refine your prompt based on the initial output.

将您的提示词输入 LLM。您可能需要根据初始输出进行迭代和优化提示词。

**Tips for interaction:**

**交互技巧：**

*   **Be patient:** LLMs can sometimes take time to process complex requests.
    **保持耐心：** LLM 有时需要时间来处理复杂的请求。
*   **Ask follow-up questions:** If you need more detail on a specific point, ask the LLM to expand on it.
    **提出后续问题：** 如果您需要某个特定点的更多细节，请要求 LLM 进行扩展。
*   **Request counter-arguments:** To prepare for a debate, ask the LLM to generate arguments for the opposing side or potential rebuttals to your points.
    **请求反驳论点：** 为了准备辩论，请要求 LLM 生成对立方的论点或对您观点的潜在驳斥。
*   **Refine the language:** If the output is too technical or not clear enough, ask the LLM to rephrase it.
    **优化语言：** 如果输出过于技术化或不够清晰，请要求 LLM 重新措辞。

### 4. Review and Refine the Output

### 4. 审查和优化输出

Once you receive the LLM's output, it's crucial to review and refine it. LLMs are powerful tools, but they are not infallible.

一旦收到 LLM 的输出，对其进行审查和优化至关重要。LLM 是强大的工具，但并非万无一失。

**What to check for:**

**需要检查的内容：**

*   **Accuracy:** Verify any facts, statistics, or claims made by the LLM.
    **准确性：** 验证 LLM 所做的任何事实、统计数据或声明。
*   **Relevance:** Ensure all points directly support the debate topic and the chosen side.
    **相关性：** 确保所有观点都直接支持辩论主题和所选的立场。
*   **Clarity and Cohesion:** Check if the arguments flow logically and are easy to understand.
    **清晰度和连贯性：** 检查论点是否逻辑清晰且易于理解。
*   **Completeness:** Does the output address all aspects of your prompt?
    **完整性：** 输出是否涵盖了您提示词的所有方面？
*   **Originality:** While LLMs generate text, it's good practice to ensure the arguments are presented in a way that feels fresh and well-reasoned.
    **原创性：** 虽然 LLM 会生成文本，但最好确保论点以一种感觉新颖且理由充分的方式呈现。

**Example Refinement Prompt:**

**示例优化提示词：**

```
Can you provide specific statistical evidence to support the claim that AI has improved healthcare diagnostics? Also, please elaborate on the potential ethical concerns related to AI in healthcare and suggest how they might be mitigated.
```

```
您能否提供具体的统计证据来支持人工智能改善了医疗诊断的说法？另外，请详细说明与医疗领域人工智能相关的潜在伦理问题，并提出如何缓解这些问题的建议。
```

### 5. Utilize the Elaborated Content

### 5. 利用详尽阐述的内容

The elaborated content from the LLM can be used in several ways:

LLM 生成的详尽阐述内容可以用于多种方式：

*   **Argument Development:** Use it as a foundation to build your own arguments and structure your debate speech.
    **论点发展：** 将其作为基础来构建您自己的论点和组织您的辩论演讲。
*   **Research:** Identify key points and areas that require further research and evidence gathering.
    **研究：** 识别需要进一步研究和收集证据的关键点和领域。
*   **Understanding Opposing Views:** Generate content for the opposing side to better anticipate and prepare rebuttals.
    **理解对立方观点：** 为对立方生成内容，以便更好地预测和准备反驳。
*   **Brainstorming:** Spark new ideas and perspectives on the topic.
    **头脑风暴：** 激发关于该主题的新想法和新视角。

By following these steps, you can effectively leverage LLMs to create comprehensive and well-reasoned elaborations on any debate topic, significantly enhancing your preparation and understanding.

通过遵循这些步骤，您可以有效地利用 LLM 为任何辩论主题创建全面且理由充分的详尽阐述，从而显著增强您的准备和理解。

In [10]:
topic_specifier_prompt = [
    SystemMessage(content="You can make a task more specific."),
    HumanMessage(
        content=f"""{game_description}
        
        You are the debate moderator.
        Please make the debate topic more specific. 
        Frame the debate topic as a problem to be solved.
        Be creative and imaginative.
        Please reply with the specified topic in {word_limit} words or less. 
        Speak directly to the presidential candidates: {*character_names,}.
        Do not add anything else."""
    ),
]
specified_topic = ChatOpenAI(temperature=1.0)(topic_specifier_prompt).content

print(f"Original topic:\n{topic}\n")
print(f"Detailed topic:\n{specified_topic}\n")

Original topic:
transcontinental high speed rail

Detailed topic:
The topic for the presidential debate is: "Overcoming the Logistics of Building a Transcontinental High-Speed Rail that is Sustainable, Inclusive, and Profitable." Donald Trump, Kanye West, Elizabeth Warren, how will you address the challenges of building such a massive transportation infrastructure, dealing with stakeholders, and ensuring economic stability while preserving the environment?



## 定义发言者选择函数
最后，我们将定义一个发言者选择函数 `select_next_speaker`，该函数接收每个代理的报价，并选择报价最高的代理（平局时随机打破）。

我们将定义一个 `ask_for_bid` 函数，该函数使用我们之前定义的 `bid_parser` 来解析代理的报价。我们将使用 `tenacity` 来装饰 `ask_for_bid`，以便在代理的报价未能正确解析时重试多次，并在达到最大尝试次数后产生一个默认报价 0。

In [11]:
@tenacity.retry(
    stop=tenacity.stop_after_attempt(2),
    wait=tenacity.wait_none(),  # No waiting time between retries
    retry=tenacity.retry_if_exception_type(ValueError),
    before_sleep=lambda retry_state: print(
        f"ValueError occurred: {retry_state.outcome.exception()}, retrying..."
    ),
    retry_error_callback=lambda retry_state: 0,
)  # Default value when all retries are exhausted
def ask_for_bid(agent) -> str:
    """
    Ask for agent bid and parses the bid into the correct format.
    """
    bid_string = agent.bid()
    bid = int(bid_parser.parse(bid_string)["bid"])
    return bid

In [12]:
import numpy as np


def select_next_speaker(step: int, agents: List[DialogueAgent]) -> int:
    bids = []
    for agent in agents:
        bid = ask_for_bid(agent)
        bids.append(bid)

    # randomly select among multiple agents with the same bid
    max_value = np.max(bids)
    max_indices = np.where(bids == max_value)[0]
    idx = np.random.choice(max_indices)

    print("Bids:")
    for i, (bid, agent) in enumerate(zip(bids, agents)):
        print(f"\t{agent.name} bid: {bid}")
        if i == idx:
            selected_name = agent.name
    print(f"Selected: {selected_name}")
    print("\n")
    return idx

## 主循环

The main loop is the core of the application. It is responsible for:

主循环是应用程序的核心。它负责：

1.  **Handling user input:** This includes keyboard presses, mouse movements, and clicks.
    1.  **处理用户输入：** 这包括键盘按键、鼠标移动和点击。
2.  **Updating the game state:** This involves moving characters, checking for collisions, and updating scores.
    2.  **更新游戏状态：** 这包括移动角色、检查碰撞以及更新分数。
3.  **Rendering the graphics:** This draws the updated game state to the screen.
    3.  **渲染图形：** 这会将更新后的游戏状态绘制到屏幕上。

Here's a simplified example of a main loop:

这是一个主循环的简化示例：

```javascript
function mainLoop() {
  // Handle user input
  handleInput();

  // Update game state
  updateGameState();

  // Render graphics
  renderGraphics();

  // Request the next frame
  requestAnimationFrame(mainLoop);
}

requestAnimationFrame(mainLoop);
```

**Explanation:**

**解释：**

-   `handleInput()`: This function is responsible for processing any user input that has occurred since the last frame.
    *   `handleInput()`：此函数负责处理自上一帧以来发生的任何用户输入。
-   `updateGameState()`: This function updates the state of the game, such as moving characters or checking for collisions.
    *   `updateGameState()`：此函数更新游戏状态，例如移动角色或检查碰撞。
-   `renderGraphics()`: This function draws the current game state to the screen.
    *   `renderGraphics()`：此函数将当前游戏状态绘制到屏幕上。
-   `requestAnimationFrame(mainLoop)`: This is a browser API that tells the browser you want to perform an animation and requests that the browser call a specified function to update an animation before the next repaint. This is the preferred way to create animations in the browser, as it allows the browser to optimize the animation and ensure it runs smoothly.
    *   `requestAnimationFrame(mainLoop)`：这是一个浏览器 API，它告诉浏览器您想要执行动画，并请求浏览器在下一次重绘之前调用指定函数来更新动画。这是在浏览器中创建动画的首选方式，因为它允许浏览器优化动画并确保其流畅运行。

In [13]:
characters = []
for character_name, character_system_message, bidding_template in zip(
    character_names, character_system_messages, character_bidding_templates
):
    characters.append(
        BiddingDialogueAgent(
            name=character_name,
            system_message=character_system_message,
            model=ChatOpenAI(temperature=0.2),
            bidding_template=bidding_template,
        )
    )

In [15]:
max_iters = 10
n = 0

simulator = DialogueSimulator(agents=characters, selection_function=select_next_speaker)
simulator.reset()
simulator.inject("Debate Moderator", specified_topic)
print(f"(Debate Moderator): {specified_topic}")
print("\n")

while n < max_iters:
    name, message = simulator.step()
    print(f"({name}): {message}")
    print("\n")
    n += 1

(Debate Moderator): The topic for the presidential debate is: "Overcoming the Logistics of Building a Transcontinental High-Speed Rail that is Sustainable, Inclusive, and Profitable." Donald Trump, Kanye West, Elizabeth Warren, how will you address the challenges of building such a massive transportation infrastructure, dealing with stakeholders, and ensuring economic stability while preserving the environment?


Bids:
	Donald Trump bid: 7
	Kanye West bid: 5
	Elizabeth Warren bid: 1
Selected: Donald Trump


(Donald Trump): Let me tell you, folks, I know how to build big and I know how to build fast. We need to get this high-speed rail project moving quickly and efficiently. I'll make sure we cut through the red tape and get the job done. And let me tell you, we'll make it profitable too. We'll bring in private investors and make sure it's a win-win for everyone. *gestures confidently*


Bids:
	Donald Trump bid: 2
	Kanye West bid: 8
	Elizabeth Warren bid: 10
Selected: Elizabeth Warren

